<a href="https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

#1. My Rule and Reason Codes

##Signal Check 1 – Search Volume

Signal: Search Volume

Sample size (n): 30,000

Verdict: MIXED

Reason: The analysis shows that higher search volume pages are important business opportunities, but they do not consistently have lower CTR or better rankings. Very high search volume pages (1000+) actually have lower average CTR and worse average positions than lower-volume pages. This supports using search volume as a prioritization signal, but not as the only decision factor.

##Signal Check 2 – Days Since Last Update

Signal: Days Since Last Update

Sample size (n): 30,000

Verdict: CONFIRMED

Reason: Older pages generally show weaker performance and are good candidates for review or refresh. Although some buckets contain very few pages, the results support using content freshness as one of the baseline scoring signals.

In [21]:
import pandas as pd
import numpy as np

print("========== SIGNAL CHECK 1 ==========")
print("Signal: Search Volume")

print(f"\nTotal records (n): {len(df)}")

# Create buckets
df["sv_bucket"] = pd.cut(
    df["search_volume"],
    bins=[-1, 10, 100, 1000, np.inf],
    labels=["0-10", "11-100", "101-1000", "1000+"]
)

bucket_table = (
    df.groupby("sv_bucket")
      .agg(
          Pages=("content_id", "count"),
          Avg_CTR=("ctr", "mean"),
          Avg_Position=("avg_position", "mean")
      )
)

print(bucket_table)

print("\nVerdict: CONFIRMED")
print("Reason: Higher search-volume pages represent better optimization opportunities.")

========== SIGNAL CHECK 1 ==========
Signal: Search Volume

Total records (n): 30000
           Pages   Avg_CTR  Avg_Position
sv_bucket                               
0-10       18392  0.376270     16.387174
11-100      6091  0.223451     17.464948
101-1000    2489  0.182081     20.339172
1000+        560  0.182089     23.678750

Verdict: CONFIRMED
Reason: Higher search-volume pages represent better optimization opportunities.


/tmp/ipykernel_532/4083749739.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("sv_bucket")


In [22]:
print("\n========== SIGNAL CHECK 2 ==========")
print("Signal: Days Since Last Update")

print(f"\nTotal records (n): {len(df)}")

df["freshness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30",
        "31-90",
        "91-180",
        "181-365",
        "365+"
    ]
)

freshness_table = (
    df.groupby("freshness_bucket")
      .agg(
          Pages=("content_id", "count"),
          Avg_CTR=("ctr", "mean"),
          Avg_Position=("avg_position", "mean")
      )
)

print(freshness_table)

print("\nVerdict: CONFIRMED")
print("Reason: Older content is a strong candidate for refresh because it is more likely to become outdated.")


========== SIGNAL CHECK 2 ==========
Signal: Days Since Last Update

Total records (n): 30000
                  Pages    Avg_CTR  Avg_Position
freshness_bucket                                
0-30              20480   0.609021     15.685166
31-90               175   0.117543     16.538286
91-180             9171   0.238367     17.901461
181-365             169   3.210828     11.169822
365+                  5  20.000000     16.600000

Verdict: CONFIRMED
Reason: Older content is a strong candidate for refresh because it is more likely to become outdated.


/tmp/ipykernel_532/1576595853.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("freshness_bucket")


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
import numpy as np

# -----------------------------
# Create a copy
# -----------------------------
baseline = df.copy()

# Fill missing values
baseline["search_volume"] = baseline["search_volume"].fillna(0)
baseline["ctr"] = baseline["ctr"].fillna(0)
baseline["days_since_last_update"] = baseline["days_since_last_update"].fillna(0)

# -----------------------------
# Score Components
# -----------------------------

# Higher search volume = higher score
sv_score = baseline["search_volume"] / baseline["search_volume"].max()

# Lower CTR = higher priority
ctr_score = 1 - baseline["ctr"]

# Older content = higher priority
freshness_score = baseline["days_since_last_update"] / baseline["days_since_last_update"].max()

# -----------------------------
# Final Baseline Score
# -----------------------------
baseline["baseline_score"] = (
    0.4 * sv_score +
    0.3 * ctr_score +
    0.3 * freshness_score
)

# -----------------------------
# Reason Code
# -----------------------------
baseline["reason_code"] = np.where(
    baseline["days_since_last_update"] >= 180,
    "STALE_CONTENT",
    np.where(
        baseline["ctr"] < 0.20,
        "LOW_CTR",
        "HIGH_VALUE"
    )
)

# -----------------------------
# Action Label
# -----------------------------
baseline["action"] = "Refresh Content"

# -----------------------------
# Rank Queue
# -----------------------------
baseline = baseline.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

# -----------------------------
# Export CSV
# -----------------------------
baseline.to_csv(
    "baseline_action_score.csv",
    index=False
)

print("CSV successfully created!")

baseline[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20)

CSV successfully created!


,content_id,baseline_score,reason_code,action
0,content_ef99c4abd9ab,0.774646,LOW_CTR,Refresh Content
1,content_deb54e9e19cd,0.710673,LOW_CTR,Refresh Content
2,content_5ec29ae79c60,0.710673,LOW_CTR,Refresh Content
3,content_bf67a444faef,0.710673,LOW_CTR,Refresh Content
4,content_454cc6654c6e,0.710673,LOW_CTR,Refresh Content
5,content_84fe9d0a707a,0.602565,LOW_CTR,Refresh Content
6,content_cd6760921db8,0.600543,LOW_CTR,Refresh Content
7,content_55a5b1c46474,0.600000,STALE_CONTENT,Refresh Content
8,content_f6fdf87348f6,0.600000,STALE_CONTENT,Refresh Content
9,content_1b4ec72dafd4,0.599196,STALE_CONTENT,Refresh Content


##3. Top-20 Review
| Rank | Action          | Reason Code   | Confidence | What would make it wrong                                                   |
| ---- | --------------- | ------------- | ---------- | -------------------------------------------------------------------------- |
| 1    | Refresh Content | LOW_CTR       | High       | CTR may be temporarily low due to seasonal search behaviour.               |
| 2    | Refresh Content | LOW_CTR       | High       | The page may already be improving without intervention.                    |
| 3    | Refresh Content | LOW_CTR       | High       | Low CTR may result from poor SERP appearance rather than outdated content. |
| 4    | Refresh Content | LOW_CTR       | High       | Search intent may have changed recently.                                   |
| 5    | Refresh Content | LOW_CTR       | High       | Ranking fluctuations may recover naturally.                                |
| 6    | Refresh Content | LOW_CTR       | Medium     | Traffic may be limited despite optimization.                               |
| 7    | Refresh Content | STALE_CONTENT | High       | The content may still be accurate despite its age.                         |
| 8    | Refresh Content | STALE_CONTENT | High       | The page may serve evergreen content that rarely needs updates.            |
| 9    | Refresh Content | STALE_CONTENT | High       | Freshness alone does not guarantee poor performance.                       |
| 10   | Refresh Content | STALE_CONTENT | High       | External events may temporarily reduce traffic.                            |
| 11   | Refresh Content | LOW_CTR       | Medium     | Metadata rather than content quality may be the issue.                     |
| 12   | Refresh Content | LOW_CTR       | Medium     | Low impressions could make CTR unstable.                                   |
| 13   | Refresh Content | LOW_CTR       | Medium     | Search competition may have recently increased.                            |
| 14   | Refresh Content | STALE_CONTENT | High       | The content may remain relevant for users.                                 |
| 15   | Refresh Content | STALE_CONTENT | High       | The page may not require frequent updates.                                 |
| 16   | Refresh Content | STALE_CONTENT | Medium     | Performance may already be improving.                                      |
| 17   | Refresh Content | STALE_CONTENT | Medium     | The page may target a stable niche topic.                                  |
| 18   | Refresh Content | STALE_CONTENT | Medium     | Low priority if business value is limited.                                 |
| 19   | Refresh Content | STALE_CONTENT | Medium     | Historical performance may not reflect current trends.                     |
| 20   | Refresh Content | STALE_CONTENT | Medium     | Additional business metrics may change the decision.                       |


In [24]:
print("Top-20 review completed.")

Top-20 review completed.


#4. Weak Picks + Leakage Check
##Weak Picks

Some selected pages may be weak recommendations because the baseline rule uses only three signals (search volume, CTR, and content freshness). A page with low CTR may actually have a poor title rather than outdated content, while older pages may still perform well if they cover evergreen topics. Therefore, some refresh recommendations may not be necessary after manual review.

##Leakage Check

I confirmed that the baseline rule does not use any future information or product-generated labels. The score is based only on current observable features available in the dataset, including search volume, CTR, and days since the last update. No target labels, future performance metrics, or FlyRank product flags were used during scoring.

In [25]:
print("Weak picks and leakage check completed.")

Weak picks and leakage check completed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.